# Background to signal $K_SKK$ and $4\pi$
## Calculate background to signal ratios for $K_SKK$ and $4\pi$ backgrounds

### Include library for handling uncertainties
#### [Here is the ```uncertainties-cpp``` library on GitHub](https://github.com/Gattocrucco/uncertainties-cpp)

In [1]:
gInterpreter->AddIncludePath("/data/lhcb/users/tat/uncertainties-cpp");

In [2]:
#include<uncertainties/impl.hpp>
#include<uncertainties/ureal.hpp>
#include<uncertainties/io.hpp>
#include<uncertainties/math.hpp>
#include<uncertainties/stat.hpp>

### Load utility functions

In [3]:
gROOT->ProcessLine(".L ../UtilityFunctions.C");

### Number of bins

In [4]:
const int NumberBins = 4;

### Get reconstructed background bin yields

In [5]:
std::map<int, double> GetRecBackgroundBinYields(const std::string &TagMode) {
    std::string Filename = "${BES3_ANALYSIS_PATH}/Selection/PeakingBackgrounds/";
    Filename += "DoubleTag/" + TagMode + "/KSKK_vs_" + TagMode + "_to_KKpipi_vs_";
    Filename += TagMode + "_DoubleTag_SignalMC_Binned.root";
    TChain Chain((TagMode + "DoubleTag").c_str());
    Chain.Add(Filename.c_str());
    return GetBinYields(&Chain, true, NumberBins);
}

### Get reconstructed background bin yields of the $D\to\pi\pi\pi\pi$ background, in bins of the $K_S\pi\pi$ side

In [6]:
std::map<int, double> GetRec4piBackgroundBinYields(const std::string &TagMode) {
    std::string Filename = "${BES3_ANALYSIS_PATH}/Selection/PeakingBackgrounds/";
    Filename += "DoubleTag/" + TagMode + "/KKpipi_vs_pipipipi_to_KKpipi_vs_";
    Filename += TagMode + "_DoubleTag_SignalMC_Binned.root";
    TChain Chain((TagMode + "DoubleTag").c_str());
    Chain.Add(Filename.c_str());
    return GetBinYields(&Chain, true, 8, "", "TagBin");
}

### Get reconstructed background yields of the $D\to K_S\pi\pi$ background in the $K_L\pi\pi$ tag

In [7]:
std::map<int, double> GetRecKSpipiToKLpipiBackgroundBinYields() {
    std::string Filename = "${BES3_ANALYSIS_PATH}/Selection/PeakingBackgrounds/";
    Filename += "DoubleTag/KLpipi/KKpipi_vs_KSpipi_to_KKpipi_vs_KLpipi";
    Filename += "_DoubleTag_SignalMC_Binned.root";
    TChain Chain("KLpipiDoubleTag");
    Chain.Add(Filename.c_str());
    return GetBinYields(&Chain, true, 8, "", "TagBin");
}

### Get generated signal yields

In [8]:
double GetGeneratedSignalYield(std::string TagMode, const std::string &WeightName = "") {
    if(TagMode.find("PartReco") != std::string::npos) {
        TagMode = TagMode.substr(0, TagMode.length() - 8);
    }
    std::string Filename = "${BES3_ANALYSIS_PATH}/TruthTuples/BinnedTruthTuples/";
    Filename += TagMode + "/KKpipi_vs_" + TagMode + "_TruthTuple_Binned.root";
    TChain Chain("TruthTuple");
    Chain.Add(Filename.c_str());
    if(WeightName.empty()) {
        return Chain.GetEntries();
    } else {
        return SumWeights(&Chain, WeightName);
    }
}

In [9]:
std::map<int, double> GetGeneratedBinYields(const std::string &TagMode) {
    std::string Filename = "${BES3_ANALYSIS_PATH}/TruthTuples/BinnedTruthTuples/";
    Filename += TagMode + "/KKpipi_vs_" + TagMode + "_TruthTuple_Binned.root";
    TChain Chain("TruthTuple");
    Chain.Add(Filename.c_str());
    return GetBinYields(&Chain, false, 8, "", "TagBin");
}

### List of tags and their backgrounds

In [10]:
// Map of tag modes, and the number label of this background
const std::vector<std::string> Tags{
    "KK",
    "KSpi0",
    "KLpi0",
    "KKPartReco",
    "KSpi0PartReco"
};

### Save parameters to a file

In [11]:
std::ofstream File("BackgroundToSignalRatios_KSKK_and_pipipipi.txt");

### Start calculating the background to signal bin efficiencies, times the ratio of branching fractions, for CP tags

In [12]:
std::string BackgroundToSignalRatios;
const auto SignalBF = GetBranchingFraction("KKpipi");
uncertainties::udouble SignalBF_unc(SignalBF.first, SignalBF.second);
const auto BackgroundBF = GetBranchingFraction("KSKK");
uncertainties::udouble BackgroundBF_unc(BackgroundBF.first, BackgroundBF.second);
const auto BFRatio = BackgroundBF_unc/SignalBF_unc;
const double BackgroundGenYields = 800000.0;
for(const auto &Tag : Tags) {
    std::vector<uncertainties::udouble> BkgToSigRatio;
    const auto SignalRecYields = GetRecSignalBinYields(Tag, NumberBins);
    const double SignalGenYields = GetGeneratedSignalYield(Tag);
    const auto BackgroundRecYields = GetRecBackgroundBinYields(Tag);
    for(int Bin = 1; Bin <= NumberBins; Bin++) {
        std::string Label = Tag + "_PeakingBackground0";
        Label += "_DoubleTag_CP_KKpipi_vs_" + Tag + "_SignalBin";
        Label += std::to_string(Bin) + "_BackgroundToSignalRatio";
        const double SigEff = SignalRecYields.at(Bin)/SignalGenYields;
        const double SigEff_err = TMath::Sqrt(SigEff*(1.0 - SigEff)/SignalGenYields);
        const uncertainties::udouble SigEff_unc(SigEff, SigEff_err);
        const double BkgEff = BackgroundRecYields.at(Bin)/BackgroundGenYields;
        const double BkgEff_err = TMath::Sqrt(BkgEff*(1.0 - BkgEff)/BackgroundGenYields);
        const uncertainties::udouble BkgEff_unc(BkgEff, BkgEff_err);
        const auto EffRatio = BkgEff_unc/SigEff_unc;
        BkgToSigRatio.push_back(EffRatio*BFRatio);
        BackgroundToSignalRatios += Label + " ";
        BackgroundToSignalRatios += std::to_string(uncertainties::nom(BkgToSigRatio.back())) + "\n";
        BackgroundToSignalRatios += Label + "_err ";
        BackgroundToSignalRatios += std::to_string(uncertainties::sdev(BkgToSigRatio.back())) + "\n";
    }
    BackgroundToSignalRatios += "\n";
    std::string Filename = "PeakingBackground_DT_KSKK_to_KKpipi_vs_" + Tag + ".root";
    std::vector<double> FlatCovMatrix =
        uncertainties::cov_matrix<std::vector<double>>(BkgToSigRatio);
    SaveCovMatrix(FlatCovMatrix, Filename);
}
File << BackgroundToSignalRatios;

### Do the same thing for $K_S\pi\pi$ tags

In [13]:
// List of tag modes, in order
const std::vector<std::string> Tags{
    "KSpipi",
    "KSpipiPartReco",
    "KLpipi"
};

In [14]:
std::string BackgroundToSignalRatios;
for(const auto &Tag : Tags) {
    std::vector<std::string> BkgToSigInOrder(8);
    std::vector<uncertainties::udouble> BkgToSigRatio;
    for(int Bin = -NumberBins; Bin <= NumberBins; Bin++) {
        if(Bin == 0) {
            continue;
        }
        const auto SignalRecYields = GetRecSignalBinYields(Tag, NumberBins);
        const double SignalGenYields = GetGeneratedSignalYield(Tag, "");
        const auto BackgroundRecYields = GetRecBackgroundBinYields(Tag);
        std::string Label = Tag + "_PeakingBackground1";
        Label += "_DoubleTag_SCMB_KKpipi_vs_" + Tag + "_SignalBin";
        Label += (Bin > 0 ? "P" : "M");
        Label += std::to_string(TMath::Abs(Bin)) + "_TagBin";
        const double SigEff = (SignalRecYields.at(Bin) + SignalRecYields.at(-Bin))/SignalGenYields;
        const double SigEff_err = TMath::Sqrt(SigEff*(1.0 - SigEff)/SignalGenYields);
        const uncertainties::udouble SigEff_unc(SigEff, SigEff_err);
        const double BkgEff = (BackgroundRecYields.at(Bin) + BackgroundRecYields.at(-Bin))/BackgroundGenYields;
        const double BkgEff_err = TMath::Sqrt(BkgEff*(1.0 - BkgEff)/BackgroundGenYields);
        const uncertainties::udouble BkgEff_unc(BkgEff, BkgEff_err);
        const auto EffRatio = BkgEff_unc/SigEff_unc;
        const auto BkgToSigRatio_unc = EffRatio*BFRatio;
        BkgToSigRatio.push_back(BkgToSigRatio_unc);
        for(int TagBin = 1; TagBin <= 8; TagBin++) {
            BkgToSigInOrder[TagBin - 1] += Label + std::to_string(TagBin) + "_BackgroundToSignalRatio" + " ";
            BkgToSigInOrder[TagBin - 1] += std::to_string(uncertainties::nom(BkgToSigRatio_unc)) + "\n";
            BkgToSigInOrder[TagBin - 1] += Label + std::to_string(TagBin) + "_BackgroundToSignalRatio_err" + " ";
            BkgToSigInOrder[TagBin - 1] += std::to_string(uncertainties::sdev(BkgToSigRatio_unc)) + "\n";
        }
    }
    for(const auto &String : BkgToSigInOrder) {
        BackgroundToSignalRatios += String;
    }
    BackgroundToSignalRatios += "\n";
    for(std::size_t i = 0; i < 7; i++) {
        for(std::size_t j = 0; j < 8; j++) {
            BkgToSigRatio.push_back(BkgToSigRatio[j]);
        }
    }
    std::string Filename = "PeakingBackground_DT_KSKK_to_KKpipi_vs_" + Tag + ".root";
    std::vector<double> FlatCovMatrix =
        uncertainties::cov_matrix<std::vector<double>>(BkgToSigRatio);
    SaveCovMatrix(FlatCovMatrix, Filename);
}


File << BackgroundToSignalRatios;

### Repeat for $4\pi$ background in $K_S\pi\pi$

In [15]:
// List of tag modes, in order
const std::vector<std::string> Tags{
    "KSpipi",
    "KSpipiPartReco",
};

In [16]:
std::string BackgroundToSignalRatios;
const auto SignalBF = GetBranchingFraction("KSpipi");
uncertainties::udouble SignalBF_unc(SignalBF.first, SignalBF.second);
const auto BackgroundBF = GetBranchingFraction("pipipipi");
uncertainties::udouble BackgroundBF_unc(BackgroundBF.first, BackgroundBF.second);
const auto BFRatio = BackgroundBF_unc/SignalBF_unc;
for(const auto &Tag : Tags) {
    std::vector<uncertainties::udouble> BkgToSigRatio;
    for(int TagBin = 1; TagBin <= 8; TagBin++) {
        const auto SignalRecYields = GetRecSignalBinYields(Tag, 8, "", "TagBin");
        const double SignalGenYields = GetGeneratedSignalYield(Tag, "");
        const auto BackgroundRecYields = GetRec4piBackgroundBinYields(Tag);
        const double SigEff = SignalRecYields.at(TagBin)/SignalGenYields;
        const double SigEff_err = TMath::Sqrt(SigEff*(1.0 - SigEff)/SignalGenYields);
        const uncertainties::udouble SigEff_unc(SigEff, SigEff_err);
        const double BkgEff = BackgroundRecYields.at(TagBin)/BackgroundGenYields;
        const double BkgEff_err = TMath::Sqrt(BkgEff*(1.0 - BkgEff)/BackgroundGenYields);
        const uncertainties::udouble BkgEff_unc(BkgEff, BkgEff_err);
        const auto EffRatio = BkgEff_unc/SigEff_unc;
        const auto BkgToSigRatio_unc = EffRatio*BFRatio;
        for(int Bin = -NumberBins; Bin <= NumberBins; Bin++) {
            if(Bin == 0) {
                continue;
            }
            BkgToSigRatio.push_back(BkgToSigRatio_unc);
            std::string Label = Tag + "_PeakingBackground0";
            Label += "_DoubleTag_SCMB_KKpipi_vs_" + Tag + "_SignalBin";
            Label += (Bin > 0 ? "P" : "M");
            Label += std::to_string(TMath::Abs(Bin)) + "_TagBin";
            BackgroundToSignalRatios += Label + std::to_string(TagBin) + "_BackgroundToSignalRatio" + " ";
            BackgroundToSignalRatios += std::to_string(uncertainties::nom(BkgToSigRatio_unc)) + "\n";
            BackgroundToSignalRatios += Label + std::to_string(TagBin) + "_BackgroundToSignalRatio_err" + " ";
            BackgroundToSignalRatios += std::to_string(uncertainties::sdev(BkgToSigRatio_unc)) + "\n";
        }
    }
    BackgroundToSignalRatios += "\n";
    std::string Filename = "PeakingBackground_DT_4pi_to_" + Tag + ".root";
    std::vector<double> FlatCovMatrix =
        uncertainties::cov_matrix<std::vector<double>>(BkgToSigRatio);
    SaveCovMatrix(FlatCovMatrix, Filename);
}


File << BackgroundToSignalRatios;

### Finally, do the same with the $K_S\pi\pi$ background in the $K_L\pi\pi$ tag

In [17]:
std::string BackgroundToSignalRatios;
// This BF is from a BESIII analysis still in review
const auto SignalBF = GetBranchingFraction("KLpipi");
uncertainties::udouble SignalBF_unc(SignalBF.first, SignalBF.second);
// Code the KSpipi BF explicitly to account for KS->pi0pi0 BF
auto BackgroundBF_unc = uncertainties::udouble(0.0280, 0.0018)*uncertainties::udouble(0.3069, 0.0005);
const auto BFRatio = BackgroundBF_unc/SignalBF_unc;
std::vector<uncertainties::udouble> BkgToSigRatio;
for(int TagBin = 1; TagBin <= 8; TagBin++) {
    const auto SignalRecYields = GetRecSignalBinYields("KLpipi", 8, "", "TagBin");
    const auto SignalGenYields = GetGeneratedBinYields("KLpipi");
    const auto BackgroundRecYields = GetRecKSpipiToKLpipiBackgroundBinYields();
    const auto BackgroundGenBinYields = GetGeneratedBinYields("KSpipi");
    const double SigEff = SignalRecYields.at(TagBin)/SignalGenYields.at(TagBin);
    const double SigEff_err = TMath::Sqrt(SigEff*(1.0 - SigEff)/SignalGenYields.at(TagBin));
    const uncertainties::udouble SigEff_unc(SigEff, SigEff_err);
    const double BkgEff = BackgroundRecYields.at(TagBin)/BackgroundGenBinYields.at(TagBin);
    const double BkgEff_err = TMath::Sqrt(BkgEff*(1.0 - BkgEff)/BackgroundGenBinYields.at(TagBin));
    const uncertainties::udouble BkgEff_unc(BkgEff, BkgEff_err);
    const auto EffRatio = BkgEff_unc/SigEff_unc;
    const auto BkgToSigRatio_unc = EffRatio*BFRatio;
    for(int Bin = -NumberBins; Bin <= NumberBins; Bin++) {
        if(Bin == 0) {
            continue;
        }
        BkgToSigRatio.push_back(BkgToSigRatio_unc);
        std::string Label = "KLpipi_PeakingBackground0";
        Label += "_DoubleTag_SCMB_KKpipi_vs_KLpipi_SignalBin";
        Label += (Bin > 0 ? "P" : "M");
        Label += std::to_string(TMath::Abs(Bin)) + "_TagBin";
        BackgroundToSignalRatios += Label + std::to_string(TagBin) + "_BackgroundToSignalRatio" + " ";
        BackgroundToSignalRatios += std::to_string(uncertainties::nom(BkgToSigRatio_unc)) + "\n";
        BackgroundToSignalRatios += Label + std::to_string(TagBin) + "_BackgroundToSignalRatio_err" + " ";
        BackgroundToSignalRatios += std::to_string(uncertainties::sdev(BkgToSigRatio_unc)) + "\n";
    }
}
std::string Filename = "PeakingBackground_DT_KSpipi_to_KLpipi.root";
std::vector<double> FlatCovMatrix =
    uncertainties::cov_matrix<std::vector<double>>(BkgToSigRatio);
SaveCovMatrix(FlatCovMatrix, Filename);
File << BackgroundToSignalRatios;

In [18]:
File.close();